# ⚙️ Split-Apply-Combine in Pandas

---

## Why This Pattern Matters

One of the most frequent tasks in data analysis is:
> *"Calculate the average life expectancy **per year**."*  
> *"Find the total sales **per region**."*  
> *"Count missing values **per column**."*

These all follow the same three-step pattern — **Split → Apply → Combine** — which is the conceptual backbone of pandas `groupby()` operations.

```
SPLIT       → divide the data into groups (e.g., by year)
APPLY       → run a function on each group (e.g., calculate mean)
COMBINE     → collect and assemble the results into a new structure
```

---

## What You Will Learn

| Section | Topics |
|---|---|
| **Part 1: Functions** | Defining reusable functions, `def`, `return`, parameters |
| **Part 2: Apply** | Applying functions to Series and DataFrames, `axis=0/1` |
| **Part 3: Lambda Functions** | Anonymous one-line functions |
| **Part 4: Split-Apply-Combine** | The full `groupby` pattern |
| **Part 5: Aggregation** | `mean`, `std`, `agg`, multi-function aggregation, dict syntax |
| **Part 6: Transform** | 1-to-1 mappings, z-score standardisation |
| **Part 7: Filter** | Removing entire groups based on group-level conditions |

In [ ]:
# Suppress deprecation and future warnings to keep notebook output clean
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

---

# Part 1: Functions — Basics

---

## 1.1 Why Functions?

A **function** is a named, reusable block of code. Instead of copying the same logic in 10 places, you define it once and call it by name anywhere you need it.

The guiding principle is **D.R.Y — Don't Repeat Yourself**.

**Anatomy of a Python function:**

```python
def function_name(parameter1, parameter2):   # 'def' keyword + name + parameters
    # Body: indented 4 spaces — Python uses indentation to define blocks
    result = parameter1 + parameter2
    return result                            # 'return' sends a value back to the caller
```

**Key rules:**
- `def` — keyword that declares a function
- `:` — colon ends the function signature
- **4-space indent** — everything in the function body must be indented
- `return` — sends a value back; without it the function returns `None`
- Functions can have zero, one, or many parameters
- Parameters can have **default values**: `def foo(x, e=2):`

---

## 1.2 Three Function Examples — Zero, One, Two Parameters

In [ ]:
# ──────────────────────────────────────────────────────
# FUNCTION 1: Zero parameters — just prints a message
# ──────────────────────────────────────────────────────
# 'def' declares the start of a function definition
# 'foo' is the function name (arbitrary — could be any valid identifier)
# '()' means no parameters are accepted
def foo():
    # This body runs every time foo() is called
    # The 4-space indent tells Python this code belongs inside the function
    print("Calling Foo")

# Call the function by writing its name followed by parentheses
# Nothing is passed in because foo() takes no parameters
foo()    # → prints: Calling Foo

# Call it multiple times — the code inside runs each time
foo()    # → prints: Calling Foo again

In [ ]:
# ──────────────────────────────────────────────────────
# FUNCTION 2: One parameter — returns a value
# ──────────────────────────────────────────────────────
# 'n' is the parameter — a local variable that receives the caller's value
def my_sq(n):
    # 'return' sends the result back to whoever called the function
    # Without 'return', the function would return None
    return n * n

# Call with n=9 → 9 * 9 = 81
result = my_sq(9)
print("my_sq(9)  =", result)      # → 81

# Call with different values
print("my_sq(4)  =", my_sq(4))    # → 16
print("my_sq(0)  =", my_sq(0))    # → 0
print("my_sq(-3) =", my_sq(-3))   # → 9 (negative squared is positive)

In [ ]:
# ──────────────────────────────────────────────────────
# FUNCTION 3: Two parameters — flexible exponentiation
# ──────────────────────────────────────────────────────
# 'x' is the base, 'e' is the exponent
# ** is Python's power operator: x ** e means x to the power of e
def my_exp(x, e):
    return x ** e

# Call with x=6, e=3 → 6³ = 216
print("my_exp(6, 3) =", my_exp(6, 3))    # → 216

# Call with named (keyword) arguments — order doesn't matter when named
print("my_exp(e=2, x=5) =", my_exp(e=2, x=5))   # → 25 (5²)

# Call with different values
print("my_exp(2, 10) =", my_exp(2, 10))  # → 1024 (2¹⁰)
print("my_exp(9, 0.5) =", my_exp(9, 0.5))  # → 3.0 (square root of 9)

---

## 1.3 Functions with Default Parameter Values

You can give parameters **default values** so callers don't always have to supply them.

```python
def my_exp(x, e=2):     # e defaults to 2 if not supplied
    return x ** e

my_exp(5)     # uses default e=2 → 5² = 25
my_exp(5, 3)  # overrides default → 5³ = 125
```

In [ ]:
# ──────────────────────────────────────────────────────
# Default parameter values make functions more flexible
# ──────────────────────────────────────────────────────
# e=2 is the default exponent — the caller can omit it
def my_exp_default(x, e=2):
    return x ** e

# Using the default (no e supplied) — computes x squared
print("my_exp_default(5)   =", my_exp_default(5))     # → 25  (5²)

# Overriding the default — computes x cubed
print("my_exp_default(5,3) =", my_exp_default(5, 3))  # → 125 (5³)

# Overriding with keyword argument
print("my_exp_default(5,e=4)=", my_exp_default(5, e=4)) # → 625 (5⁴)

---

# Part 2: Apply — Running Functions Across DataFrames

---

## 2.1 What is `apply()`?

`apply()` lets you run any Python function across every element of a **Series**, or across every column/row of a **DataFrame**.

This is more flexible than built-in methods like `.sum()` or `.mean()` because you can pass **any custom function**.

**Two contexts:**

| Context | What is passed to the function | `axis` parameter |
|---|---|---|
| **Series** | Each individual value in the Series | Not applicable |
| **DataFrame, axis=0** (default) | Each **column** as a Series | `axis=0` or `axis='index'` |
| **DataFrame, axis=1** | Each **row** as a Series | `axis=1` or `axis='columns'` |

**Key constraint:** The function must accept **at least one argument** (the element or Series being passed). Zero-argument functions cannot be used with `.apply()`.

---

## 2.2 Build a Sample DataFrame

In [ ]:
# Import pandas for DataFrame operations
import pandas as pd

# Create a dictionary to define the DataFrame columns
# Each key becomes a column name; each list becomes that column's values
d = {
    'a': [10, 20, 30],   # Column 'a' has 3 values
    'b': [20, 30, 40]    # Column 'b' has 3 values
}

# Build the DataFrame from the dictionary
df = pd.DataFrame(data=d)

# Display shows the full table in Jupyter; print works too
print("Our sample DataFrame:")
print(df)
print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")

---

## 2.3 Apply Over a Series (Column)

When you call `.apply()` on a **single column** (a Series), pandas passes each cell value individually to your function — like a loop.

```
df['a'] = [10, 20, 30]

df['a'].apply(my_sq):
  → my_sq(10) = 100
  → my_sq(20) = 400
  → my_sq(30) = 900
  → Returns new Series: [100, 400, 900]
```

In [ ]:
# ──────────────────────────────────────────────────────
# WHY zero-argument functions don't work with apply
# ──────────────────────────────────────────────────────
# apply() MUST pass a value to the function
# foo() takes NO arguments → error because apply has nowhere to send the value
# Uncomment the next line to see the TypeError:
# df['a'].apply(foo)   # → TypeError: foo() takes 0 positional arguments but 1 was given
print("Zero-argument functions cannot be used with apply() — they receive no value.")

print()

# ──────────────────────────────────────────────────────
# Applying a single-parameter function to a Series
# ──────────────────────────────────────────────────────
# df['a'] is a Series: [10, 20, 30]
# .apply(func=my_sq): passes each element (10, then 20, then 30) to my_sq
# Returns a NEW Series with the squared values — df itself is NOT modified
print("Applying my_sq to column 'a':")
squared = df['a'].apply(func=my_sq)
print(squared)
# → 0: 100  (10² = 100)
# → 1: 400  (20² = 400)
# → 2: 900  (30² = 900)

print()
# Confirm the original DataFrame is unchanged — apply returns a new object
print("Original DataFrame (unchanged):")
print(df)

In [ ]:
# ──────────────────────────────────────────────────────
# Applying a multi-parameter function to a Series
# ──────────────────────────────────────────────────────
# my_exp(x, e) takes TWO parameters:
#   - x: the value from the Series (passed automatically by apply)
#   - e: an additional argument we supply manually as a keyword argument
# apply() passes the Series element as the FIRST positional argument
# Extra arguments (e=3) are passed through as keyword arguments
print("Applying my_exp with e=3 to column 'a' (each value cubed):")
cubed = df['a'].apply(func=my_exp, e=3)
print(cubed)
# → 0: 1000  (10³ = 1000)
# → 1: 8000  (20³ = 8000)
# → 2: 27000 (30³ = 27000)

print()
# Apply to the other column with a different exponent
print("Applying my_exp with e=0.5 to column 'b' (square root):")
sqrt_b = df['b'].apply(func=my_exp, e=0.5)
print(sqrt_b.round(3))
# → square roots of 20, 30, 40

---

## 2.4 Apply Over a Full DataFrame — Column-wise vs Row-wise

When `.apply()` is called on a **DataFrame**, the `axis` parameter controls the direction:

```
axis=0  (default):  The function receives each COLUMN as a Series
                    → Operates down the rows within each column
                    → Results in one output per column

axis=1:             The function receives each ROW as a Series
                    → Operates across the columns within each row
                    → Results in one output per row
```

Visual intuition:
```
    a   b           axis=0: function sees [10,20,30], then [20,30,40]
0  10  20           axis=1: function sees [10,20], then [20,30], then [30,40]
1  20  30
2  30  40
```

In [ ]:
# Define a helper function that simply prints what it receives
# This makes it easy to SEE what apply() passes to the function in each direction
def print_me(x):
    # x will be a pandas Series — either a column or a row
    print(x)
    print("---")   # separator for readability

In [ ]:
# ──────────────────────────────────────────────────────
# axis=0: Column-wise — function receives each COLUMN
# ──────────────────────────────────────────────────────
# With axis=0, apply() loops through the COLUMNS
# First call: x = column 'a' = Series([10, 20, 30], name='a')
# Second call: x = column 'b' = Series([20, 30, 40], name='b')
print("=== axis=0: Function receives each COLUMN ===")
df.apply(func=print_me, axis=0)

In [ ]:
# ──────────────────────────────────────────────────────
# axis=1: Row-wise — function receives each ROW
# ──────────────────────────────────────────────────────
# With axis=1, apply() loops through the ROWS
# First call:  x = row 0 = Series({'a': 10, 'b': 20}, name=0)
# Second call: x = row 1 = Series({'a': 20, 'b': 30}, name=1)
# Third call:  x = row 2 = Series({'a': 30, 'b': 40}, name=2)
print("=== axis=1: Function receives each ROW ===")
df.apply(func=print_me, axis=1)

---

## 2.5 Practical Apply — Missing Data Analysis on Titanic Dataset

A realistic use of `.apply()`: calculate how complete each column is in a dataset.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from io import StringIO

# Build a Titanic-like dataset in memory (simulates reading titanic.csv)
# In real usage: df_titanic = pd.read_csv('../Data/titanic.csv')
titanic_data = """survived,pclass,name,sex,age,sibsp,parch,fare,cabin,embarked
0,3,Braund,male,22.0,1,0,7.25,,S
1,1,Cumings,female,38.0,1,0,71.28,C85,C
1,3,Heikkinen,female,26.0,0,0,7.92,,S
1,1,Futrelle,female,35.0,1,0,53.10,C123,S
0,3,Allen,male,35.0,0,0,8.05,,S
0,3,Moran,male,,0,0,8.46,,Q
0,1,McCarthy,male,54.0,0,0,51.86,E46,S
0,3,Palsson,male,2.0,3,1,21.07,,S
1,3,Johnson,female,27.0,0,2,11.13,,S
1,2,Nasser,female,14.0,1,0,30.07,,C"""

# Read the CSV string into a DataFrame
df_titanic = pd.read_csv(StringIO(titanic_data))
print("Titanic dataset preview:")
print(df_titanic.to_string())
print(f"\nShape: {df_titanic.shape}")

In [ ]:
# ──────────────────────────────────────────────────────
# Define three utility functions for missing data analysis
# These will be applied column-wise to the Titanic DataFrame
# ──────────────────────────────────────────────────────

# Function 1: count_missing — counts null values in a Series
# pd.isnull(ser) returns a boolean Series (True = missing)
# np.sum(...) adds up all the Trues (each True counts as 1)
def count_missing(ser):
    nulls = pd.isnull(ser)    # boolean mask: True where value is NaN
    return np.sum(nulls)       # count the True values = count of missing

# Function 2: prop_missing — proportion of missing values (0.0 to 1.0)
# Divides the count of missing by total number of elements in the Series
def prop_missing(ser):
    num = count_missing(ser)   # count of missing values (numerator)
    den = ser.size             # total elements including NaN (denominator)
    return num / den           # proportion: e.g., 0.3 = 30% missing

# Function 3: prop_complete — proportion that is NOT missing
# Simply the complement of prop_missing: 1 - (proportion missing)
def prop_complete(ser):
    return 1 - prop_missing(ser)  # e.g., if 30% missing → 70% complete

In [ ]:
# ──────────────────────────────────────────────────────
# Apply count_missing COLUMN-WISE (axis=0, the default)
# ──────────────────────────────────────────────────────
# axis=0 (default): each column is passed to count_missing as a Series
# Returns a Series showing how many missing values are in each column
print("=== Missing value COUNT per column (axis=0 default) ===")
missing = df_titanic.apply(func=count_missing)  # axis=0 is default
print(missing)
print("\nColumns with missing data:", missing[missing > 0].index.tolist())

In [ ]:
# ──────────────────────────────────────────────────────
# Apply prop_missing COLUMN-WISE (explicit axis=0)
# ──────────────────────────────────────────────────────
# Same as above but returns proportions (0.0–1.0) rather than counts
# Multiply by 100 to get percentage
print("=== Proportion MISSING per column ===")
p_missing = df_titanic.apply(func=prop_missing, axis=0)
print((p_missing * 100).round(1).astype(str) + '%')

In [ ]:
# ──────────────────────────────────────────────────────
# Apply prop_complete ROW-WISE (axis=1)
# ──────────────────────────────────────────────────────
# axis=1: each ROW is passed to prop_complete as a Series
# Each row has 10 columns — how many are complete (not NaN)?
# Returns a Series: one completeness score (0.0–1.0) per passenger
print("=== Proportion COMPLETE per row (each passenger) ===")
p_complete = df_titanic.apply(func=prop_complete, axis=1)
print(p_complete)
print()
print("Interpretation:")
print("  1.0 = all 10 fields present for this passenger")
print("  0.9 = 9 of 10 fields present (1 missing)")
print("  0.8 = 8 of 10 fields present (2 missing)")

---

# Part 3: Lambda Functions

---

## 3.1 What is a Lambda?

A **lambda** is an **anonymous one-line function** — a function without a name, written inline.

**Syntax:**
```python
lambda parameter1, parameter2: expression
```

**Comparison — named function vs lambda:**
```python
# Named function
def my_sq(n):
    return n * n

# Equivalent lambda
lambda n: n * n
```

**When to use lambdas:**
- Short, one-off functions that are used in only one place
- When passing a function as an argument (e.g., to `.apply()`, `.filter()`, `.sort()`)
- When naming the function would add more clutter than value

**When NOT to use lambdas:**
- Complex logic that needs multiple lines
- Functions used in more than one place (define a proper named function instead)

In [ ]:
# ──────────────────────────────────────────────────────
# Lambda basics
# ──────────────────────────────────────────────────────

# Lambda with one parameter — square a number
# 'lambda n:' defines the parameter; 'n * n' is the expression that gets returned
sq = lambda n: n * n
print("sq(9)   =", sq(9))    # → 81
print("sq(4)   =", sq(4))    # → 16

# Lambda with two parameters — exponentiation
# Both parameters are separated by a comma before the colon
exp = lambda x, e: x ** e
print("exp(6,3)=", exp(6, 3)) # → 216

# Lambda with a condition (ternary expression)
# Returns 'adult' if age >= 18, else 'minor'
classify = lambda age: 'adult' if age >= 18 else 'minor'
print("classify(25):", classify(25))  # → 'adult'
print("classify(15):", classify(15))  # → 'minor'

print()
# ──────────────────────────────────────────────────────
# Using lambda directly with apply
# ──────────────────────────────────────────────────────
# Inline lambda — no need to define a separate named function
# This does the same as: df['a'].apply(my_sq)
print("Using lambda to square column 'a':")
print(df['a'].apply(lambda n: n * n))

print()
# Lambda that checks if a value is greater than 15
print("Values in 'a' greater than 15:")
print(df['a'].apply(lambda x: x > 15))

---

# Part 4: Split-Apply-Combine — The Full Pattern

---

## 4.1 The Three Steps Explained

The **Split-Apply-Combine** strategy is how pandas implements group-based analysis:

```
ORIGINAL DATA
┌────────────┬──────────┐
│  year      │  lifeExp │
├────────────┼──────────┤
│  1952      │  28.8    │
│  1952      │  55.2    │  ← GROUP 1952
│  1952      │  62.5    │
│  1957      │  30.3    │
│  1957      │  59.3    │  ← GROUP 1957
│  1957      │  64.4    │
└────────────┴──────────┘
        │
  STEP 1: SPLIT by 'year'
        │
        ├──── Group 1952: [28.8, 55.2, 62.5]
        └──── Group 1957: [30.3, 59.3, 64.4]
                │
          STEP 2: APPLY mean() to each group
                │
                ├──── mean([28.8, 55.2, 62.5]) = 48.8
                └──── mean([30.3, 59.3, 64.4]) = 51.3
                        │
                  STEP 3: COMBINE results
                        │
                   year  lifeExp_mean
                   1952  48.8
                   1957  51.3
```

## 4.2 The Three Apply Types

After splitting, you can apply three kinds of operations:

| Apply Type | Description | Input → Output shape | Example |
|---|---|---|---|
| **Aggregation** | Reduce many values to one | N rows → 1 value per group | `mean()`, `sum()`, `count()` |
| **Transformation** | Map each value to a new value | N rows → N rows | z-score, normalise, fill NaN |
| **Filtration** | Keep or discard entire groups | N rows → ≤N rows | remove groups with < 30 rows |

---

# Part 5: Aggregation

---

## 5.1 What is Aggregation?

**Aggregation** takes multiple values and reduces them to a **single summary value**.

Also called *summarisation* or *reduction*.

Examples:
- 244 tip values → 1 mean tip per day of the week
- 1704 life expectancy values → 1 mean per year
- All salary rows → total payroll per department

## 5.2 Built-in Pandas Aggregation Functions

After `.groupby()`, you can chain any of these directly:

| Function | What it calculates |
|---|---|
| `count()` | Number of non-null values |
| `size()` | Total group size (including NaN) |
| `mean()` | Arithmetic mean |
| `median()` | Middle value |
| `std()` | Standard deviation |
| `var()` | Variance |
| `sem()` | Standard error of the mean |
| `min()` / `max()` | Minimum / maximum |
| `sum()` | Total sum |
| `first()` / `last()` | First / last value in group |
| `describe()` | Full stats summary |

In [ ]:
# Import pandas and numpy for data manipulation
import pandas as pd
import numpy as np
from io import StringIO

# Build a gapminder-style dataset in memory
# In real usage: df = pd.read_csv('../Data/gapminder.tsv', sep='\t')
gap_data = """country,continent,year,lifeExp,pop,gdpPercap
Afghanistan,Asia,1952,28.801,8425333,779.45
Afghanistan,Asia,1957,30.332,9240934,820.85
Albania,Europe,1952,55.230,1282697,1601.06
Albania,Europe,1957,59.280,1476505,1942.28
Argentina,Americas,1952,62.485,17876956,5911.32
Argentina,Americas,1957,64.399,19610538,6856.86
Australia,Oceania,1952,69.120,8691212,10039.60
Australia,Oceania,1957,70.330,9712569,10949.65
Belgium,Europe,1952,68.000,8730405,8343.11
Belgium,Europe,1957,69.240,8989111,9714.96"""

# Parse the CSV string into a DataFrame
df = pd.read_csv(StringIO(gap_data))
print("Gapminder dataset preview:")
print(df.to_string())

In [ ]:
# ──────────────────────────────────────────────────────
# Built-in pandas aggregation functions
# ──────────────────────────────────────────────────────

# groupby('year'): SPLIT the data into two groups — year 1952, year 1957
# ['lifeExp']: select only the lifeExp column from each group
# .mean(): APPLY mean to each group, COMBINE into a result Series
print("Mean life expectancy per year:")
mean_by_year = df.groupby(by='year')['lifeExp'].mean()
print(mean_by_year)

print()
# .std(): standard deviation — how spread out the values are within each year
print("Std deviation of life expectancy per year:")
std_by_year = df.groupby(by='year')['lifeExp'].std()
print(std_by_year.round(3))

print()
# .sem(): standard error of the mean — measures precision of the mean estimate
# SEM = std / sqrt(n): smaller SEM → more confidence in the mean
print("Standard error of the mean per year:")
sem_by_year = df.groupby(by='year')['lifeExp'].sem()
print(sem_by_year.round(3))

---

## 5.3 Non-Pandas Functions — Using `.agg()`

`.agg()` (short for `.aggregate()`) lets you apply **any function** to grouped data — not just pandas built-ins. This includes NumPy functions, SciPy functions, or your own custom functions.

In [ ]:
# Import numpy and scipy.stats for non-pandas aggregation functions
import numpy as np
import scipy.stats

print("=== Using numpy functions via .agg() ===")

# np.count_nonzero: counts values that are NOT zero (useful for counting non-missing)
# Here it effectively counts how many countries we have data for per year
print("Count (non-zero) per year:")
count_result = df.groupby(by='year')['lifeExp'].agg(np.count_nonzero)
print(count_result)

print()
# np.std: numpy's standard deviation (uses N in denominator, pandas uses N-1)
print("NumPy std per year (population std, divides by N):")
std_result = df.groupby(by='year')['lifeExp'].agg(np.std)
print(std_result.round(4))

print()
# scipy.stats.sem: standard error of the mean from scipy
print("SciPy SEM per year:")
sem_result = df.groupby(by='year')['lifeExp'].agg(scipy.stats.sem)
print(sem_result.round(4))

---

## 5.4 User-Defined Aggregation Functions

In [ ]:
# ──────────────────────────────────────────────────────
# User-defined aggregation: simple mean
# ──────────────────────────────────────────────────────
# agg() passes the entire GROUP as a Series to this function
# (unlike apply() on a column which passes one value at a time)
def my_mean(values):
    # 'values' is a pandas Series containing all values in the group
    n = len(values)          # total count of values in this group
    total = 0
    for v in values:         # iterate through each value in the group
        total += v           # accumulate the sum
    return total / n         # return the arithmetic mean

# Apply our custom mean to each year group's lifeExp values
print("Custom mean (my_mean) per year:")
custom_mean = df.groupby(by='year')['lifeExp'].agg(my_mean)
print(custom_mean)

print()
# Verify it matches pandas' built-in mean
print("Pandas built-in mean (should match above):")
print(df.groupby(by='year')['lifeExp'].mean())

In [ ]:
# ──────────────────────────────────────────────────────
# User-defined aggregation: function with extra parameters
# ──────────────────────────────────────────────────────
# my_mean_diff: calculates group mean MINUS a reference value
# 'values' = the group Series (passed by agg automatically)
# 'diff_value' = extra parameter we supply manually
def my_mean_diff(values, diff_value):
    n = len(values)          # number of values in the group
    total = 0
    for v in values:         # sum all values
        total += v
    mean = total / n         # group mean
    return mean - diff_value # deviation from the reference value

# Calculate the global average life expectancy across ALL years and countries
# This serves as our reference/baseline for comparison
global_mean = df['lifeExp'].mean()
print(f"Global mean life expectancy: {global_mean:.3f} years")

print()
# Apply with an extra keyword argument: diff_value=global_mean
# This shows whether each year's average is above or below the global average
# Note: when passing extra args to agg, use positional/keyword syntax (not func=)
print("Deviation of each year's mean from global mean:")
deviations = df.groupby('year')['lifeExp'].agg(my_mean_diff, diff_value=global_mean)
print(deviations.round(3))
print("\nPositive = year is above global mean, Negative = below")

---

## 5.5 Multiple Aggregation Functions at Once

In [ ]:
# ──────────────────────────────────────────────────────
# Option 1: Pass a LIST of functions to agg()
# ──────────────────────────────────────────────────────
# Each function is applied to the same group
# Result is a DataFrame with one column per function

# Mix of string names (pandas built-ins) and function references
print("=== Multiple functions as a named-functions list ===")
func_list = ['mean', 'std', 'count', 'min', 'max']
result_list = df.groupby('year')['lifeExp'].agg(func_list)
print(result_list.round(3))

print()
# ──────────────────────────────────────────────────────
# Option 2: Named aggregations with custom output column names
# ──────────────────────────────────────────────────────
# pandas.NamedAgg lets you give each result column a meaningful name
print("=== Named aggregations (custom output column names) ===")
result_named = df.groupby('year')['lifeExp'].agg(
    avg_life_exp='mean',    # output column 'avg_life_exp' = mean
    spread='std',           # output column 'spread'        = std deviation
    country_count='count',  # output column 'country_count' = count
    best='max',             # output column 'best'           = maximum
    worst='min'             # output column 'worst'          = minimum
)
print(result_named.round(3))

In [ ]:
# ──────────────────────────────────────────────────────
# Option 3: Dict syntax — different function for each column
# ──────────────────────────────────────────────────────
# Keys: column names in the DataFrame
# Values: function to apply to that column
# This lets you aggregate DIFFERENT columns DIFFERENTLY in one call

# Dictionary: which aggregation to apply to which column
func_dict = {
    'lifeExp':   'mean',    # average life expectancy per year
    'pop':       np.std,    # spread of population sizes per year
    'gdpPercap': my_mean    # custom mean of GDP per capita per year
}

# aggregate() is just a longer spelling of agg() — identical behaviour
print("=== Different function per column via dict ===")
result_dict = df.groupby('year').aggregate(func_dict)
print(result_dict.round(2))
print("\nNotice: each column used a different aggregation function")

---

# Part 6: Transform

---

## 6.1 What is Transform?

**Transform** is different from aggregation:
- **Aggregation**: many values → one result per group (reduces shape)
- **Transform**: each value → one new value, **same shape as input** (no reduction)

Transform is used for operations like:
- **Z-score standardisation**: express each value as 'how many standard deviations from the group mean'
- **Group-mean subtraction**: centre values within each group
- **Filling NaN within groups**: replace missing values with the group mean
- **Percentage of group total**: express each value as % of its group's sum

### The Z-Score

The z-score measures how far a value is from its group's mean, in units of standard deviations:

$$z = \frac{x - \mu}{\sigma}$$

- $x$ = the individual value
- $\mu$ = the group mean
- $\sigma$ = the group standard deviation

**Interpretation:**
- z = 0 → exactly at the group mean
- z = 1 → one standard deviation above the mean
- z = -2 → two standard deviations below the mean

In [ ]:
# ──────────────────────────────────────────────────────
# Define a z-score transformation function
# ──────────────────────────────────────────────────────
# This function receives a group's values as a Series
# It returns a new Series of the same length (no reduction)
def my_zscore(x):
    # x.mean(): mean of the group
    # x.std(): standard deviation of the group
    # Formula: z = (value - group_mean) / group_std
    return (x - x.mean()) / x.std()

# Apply the z-score transformation WITHIN each year group
# For each country, we get its z-score relative to its year's average
# This tells us: 'was this country above or below average FOR THAT YEAR?'
print("=== Z-score transform of lifeExp within each year group ===")
zscore_result = df.groupby(by='year')['lifeExp'].transform(my_zscore)
print(zscore_result.round(3))

print()
# KEY DIFFERENCE: transform preserves the original shape
print(f"Original shape: {df['lifeExp'].shape}")
print(f"After transform: {zscore_result.shape}")   # SAME shape
print("(Unlike agg, transform does NOT reduce the number of rows)")

print()
# Add the z-scores as a new column alongside the original data
df_result = df[['country', 'year', 'lifeExp']].copy()
df_result['lifeExp_zscore'] = zscore_result
print("Life expectancy with z-scores:")
print(df_result.sort_values('year').round(3).to_string())

In [ ]:
# ──────────────────────────────────────────────────────
# Another transform: fill NaN within groups using group mean
# ──────────────────────────────────────────────────────
# This is more useful than global mean imputation because it
# uses the group's own context to fill missing values

# Create a copy with some intentional missing values
df_missing = df[['country', 'year', 'lifeExp']].copy()
# Manually set some values to NaN to simulate missing data
df_missing.loc[df_missing['country'].isin(['Albania', 'Australia']), 'lifeExp'] = np.nan
print("Dataset with missing lifeExp values:")
print(df_missing.to_string())

print()
# Fill NaN with the GROUP mean (mean for that year)
# transform passes each year group's values to the lambda
# fillna(x.mean()) replaces NaN with the group's mean
df_missing['lifeExp_filled'] = df_missing.groupby('year')['lifeExp'].transform(
    lambda x: x.fillna(x.mean())   # fill each group's NaN with that group's mean
)
print("After group-mean imputation:")
print(df_missing.round(3).to_string())

---

# Part 7: Filter

---

## 7.1 What is GroupBy Filter?

**Filter** (in the groupby context) keeps or removes **entire groups** based on a condition evaluated at the **group level** — not at the individual row level.

The function you pass to `.filter()` must:
- Accept a group DataFrame (or Series) as its argument
- Return `True` to **keep** the group, `False` to **discard** it

**Key distinction:**
- Row-level filter: `df[df['age'] > 30]` — keeps individual rows where the condition is true
- Group-level filter: keeps ALL rows from groups that meet a group-wide condition

```
Before filter:               After filter (groups with < 30 rows removed):
size=1: 3 rows          →    REMOVED (only 3 rows)
size=2: 156 rows        →    KEPT    (156 rows ≥ 30)
size=3: 38 rows         →    KEPT    (38 rows ≥ 30)
size=4: 37 rows         →    KEPT    (37 rows ≥ 30)
size=5: 5 rows          →    REMOVED (only 5 rows)
size=6: 4 rows          →    REMOVED (only 4 rows)
```

In [ ]:
# Build a tips-like dataset with table sizes
# In real usage: df_tips = pd.read_csv('../Data/tips.csv')
tips_data = """ID,total_bill,tip,gender,day,size
1,16.99,1.01,Female,Sun,2
2,10.34,1.66,Male,Sun,3
3,21.01,3.50,Male,Sun,3
4,23.68,3.31,Male,Sun,2
5,24.59,3.61,Female,Sun,4
6,25.29,4.71,Male,Sun,4
7,8.77,2.00,Male,Sun,2
8,26.88,3.12,Male,Sun,4
9,15.04,1.96,Male,Sun,2
10,14.78,3.23,Male,Sun,2
11,10.27,1.71,Male,Sun,2
12,35.26,5.00,Female,Sun,4
13,15.42,1.57,Male,Sun,2
14,18.43,3.00,Male,Sun,4
15,14.83,3.02,Female,Sun,2
16,21.58,3.92,Male,Sun,2
17,10.33,1.67,Female,Sun,3
18,16.29,3.71,Male,Sun,3
19,16.97,3.50,Female,Sun,3
20,20.65,3.35,Male,Sat,3
21,17.92,4.08,Male,Sat,2
22,20.29,2.75,Female,Sat,2
23,15.77,2.23,Female,Sat,2"""

# Parse the dataset into a DataFrame
df_tips = pd.read_csv(StringIO(tips_data))

# Examine the distribution of table sizes — how many times each size appears
print("Distribution of table sizes (count of each size):")
print(df_tips['size'].value_counts().sort_index())
print(f"\nTotal rows: {len(df_tips)}")

In [ ]:
# ──────────────────────────────────────────────────────
# Option 1: GroupBy filter with a named function
# ──────────────────────────────────────────────────────
# gt3 receives each GROUP (a sub-DataFrame of df_tips) as 'group_df'
# It must return True (keep this group) or False (discard this group)
# Here: keep only groups (table sizes) with 3 or more observations
def gt3(group_df):
    # count() on a column returns the number of non-null values in the group
    # We check if the group has at least 3 rows
    return group_df['size'].count() >= 3

# groupby('size'): split df_tips into groups by table size
# .filter(gt3): apply gt3 to each group — keep those where gt3 returns True
# Result: a NEW DataFrame containing only the rows from kept groups
print("=== Option 1: Named function filter (keep groups with ≥ 3 rows) ===")
df_filtered = df_tips.groupby(by='size').filter(gt3)

print("Sizes remaining after filter:")
print(df_filtered['size'].value_counts().sort_index())
print(f"\nRows before: {len(df_tips)}, Rows after: {len(df_filtered)}")

In [ ]:
# ──────────────────────────────────────────────────────
# Option 2: GroupBy filter with a lambda function
# ──────────────────────────────────────────────────────
# Identical logic as Option 1, but written as an inline lambda
# 'x' receives each group's sub-DataFrame
# The lambda returns True if the group has >= 3 rows
print("=== Option 2: Lambda filter (identical result) ===")
df_filtered_lambda = df_tips.groupby(by='size').filter(
    lambda x: x['size'].count() >= 3
)

print("Sizes remaining:")
print(df_filtered_lambda['size'].value_counts().sort_index())

print()
print("Are the two approaches identical?", df_filtered.equals(df_filtered_lambda))

print()
# Advanced filter: keep groups where the MEAN tip is above the global mean
global_mean_tip = df_tips['tip'].mean()
print(f"=== Filter: keep table sizes where mean tip > global mean ({global_mean_tip:.2f}) ===")
df_high_tip_sizes = df_tips.groupby('size').filter(
    lambda x: x['tip'].mean() > global_mean_tip
)
print("Sizes with above-average tipping:")
print(df_high_tip_sizes.groupby('size')['tip'].mean().round(2))

---

# Summary

---

## Key Concepts at a Glance

### Part 1: Functions

| Concept | Syntax | Notes |
|---|---|---|
| Define a function | `def name(params): ...` | Body indented 4 spaces |
| Return a value | `return expression` | Without `return`, function returns `None` |
| Default parameter | `def f(x, e=2):` | Caller can omit `e` |
| Lambda | `lambda x: x * x` | Inline anonymous function |

### Part 2: Apply

| Context | Syntax | What function receives |
|---|---|---|
| Series | `series.apply(func)` | Each individual element |
| DataFrame columns | `df.apply(func, axis=0)` | Each column as a Series |
| DataFrame rows | `df.apply(func, axis=1)` | Each row as a Series |
| Extra args | `series.apply(func, arg=val)` | Passed through to function |

### Parts 5–7: Split-Apply-Combine

| Step | Method | Output shape |
|---|---|---|
| Split | `df.groupby('col')` | GroupBy object |
| Aggregate | `.agg('mean')` or `.mean()` | One row per group |
| Transform | `.transform(func)` | Same shape as input |
| Filter | `.filter(func)` | ≤ original rows |

---

## Self-Test Questions

1. What does D.R.Y stand for and why is it important?
2. Why can't a zero-argument function be used with `.apply()`?
3. What is the difference between `axis=0` and `axis=1` in `.apply()`?
4. What is the difference between `.agg()` and `.transform()` in a groupby context?
5. Write a lambda that returns `True` if a number is greater than 50.
6. What does the z-score formula measure? When would you use group-level z-scores?
7. How does GroupBy `.filter()` differ from regular row-level filtering with `df[condition]`?